# KG-Commit: Online Knowledge Graph Simulation

Simulates the real deployment scenario: commits arrive one-by-one in chronological order.
For each commit the loop does exactly **four steps in order**:

```
1. extract_kg_features(G, commit)   ← query G BEFORE this commit is known
2. predict(features)                ← make a prediction
3. evaluate(prediction, true_label) ← record the result
4. update_kg(G, commit)             ← add this commit's knowledge to G
```

`G` at step 1 only knows about commits **strictly before** the current one — zero label leakage.

---

### What gets added to G per commit (by tier)

| Tier | Entities | Needs |
|------|----------|-------|
| Core | `COMMIT` `TIME` `INTERVAL` `LABEL` | local CSV |
| File | `AUTHOR` `FILE` `FILE_TYPE` `DIR` `EXTERNAL_PACKAGE` | diff_text (server) |
| Within-file | `CLASS` `FUNCTION` `FUNCTION_SIGNATURE` `VARIABLE` `DATA_TYPE` | diff_text (server) |
| Finer | `ISSUE` `BRANCH` | git repo (server) |

### KG features extracted before each commit

| Feature | Source in G |
|---------|-------------|
| `kg_project_commit_count` | count of COMMIT nodes so far |
| `kg_project_bug_rate` | fraction of past commits labelled buggy |
| `kg_author_commit_count` | AUTHOR node counter |
| `kg_author_bug_rate` | AUTHOR node counter |
| `kg_file_change_count` | sum of FILE node counters for touched files |
| `kg_file_bug_rate` | mean bug rate across touched files |
| `kg_file_unique_authors` | mean unique-author count across touched files |

## 0 — Imports & Configuration

In [ ]:
import networkx as nx
import pandas as pd
import re
import json
import numpy as np
from pathlib import Path
from collections import defaultdict
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

# ── Local paths (work on any machine) ─────────────────────────────────
PROJECT_NAME = "apache_groovy"
LOCAL_CSV    = Path("../../data/apachejit/projects/apache_groovy.csv")

# ── SERVER-SIDE PLACEHOLDERS — set these before running on the server ─
DIFF_TEXT_CSV = Path("/path/on/server/diff/apache_groovy_diff.csv")
REPO_PATH     = Path("/path/on/server/repos/apache_groovy")

print(f"NetworkX  : {nx.__version__}")
print(f"Local CSV : {LOCAL_CSV.exists()}")

## 1 — Load Data

In [ ]:
# Sort by author_date — this is the commit stream order
df = pd.read_csv(LOCAL_CSV).sort_values("author_date").reset_index(drop=True)
print(f"Commits : {len(df):,}  |  {df.author_date.min()} → {df.author_date.max()}")

# diff_text CSV is server-side; fall back to empty dict if not present
if DIFF_TEXT_CSV.exists():
    _diff_df = pd.read_csv(DIFF_TEXT_CSV)
    diff_map = dict(zip(_diff_df.commit_id, _diff_df.diff_text))
    print(f"diff_text : {len(diff_map):,} commits loaded")
else:
    diff_map = {}
    print("diff_text : not found — file/author/within-file tiers will be skipped")

## 2 — diff_text Parser Functions

Pure functions — no side effects on `G`.
Each extracts one type of information from a raw `git show` string.

In [ ]:
_RE_FILE    = re.compile(r'^diff --git a/(.+?) b/(.+?)$', re.MULTILINE)
_RE_AUTHOR  = re.compile(r'^Author:\s+(.+?)\s+<(.+?)>', re.MULTILINE)
_RE_PY_IMP  = re.compile(r'^\+(?:from\s+([\w.]+)\s+import|import\s+([\w.]+))', re.MULTILINE)
_RE_JV_IMP  = re.compile(r'^\+import\s+([\w.]+);', re.MULTILINE)
_RE_CLASS   = re.compile(r'^\+\s*(?:public\s+|abstract\s+|final\s+)*class\s+(\w+)', re.MULTILINE)
_RE_PY_FN   = re.compile(r'^\+\s*(?:async\s+)?def\s+(\w+)\s*\(([^)]*)\)(?:\s*->\s*(\S+))?', re.MULTILINE)
_RE_JV_FN   = re.compile(r'^\+\s*(?:public|private|protected|static|\s)+(\w[\w<>\[\]]*)\s+(\w+)\s*\(([^)]*)\)', re.MULTILINE)
_RE_PY_VAR  = re.compile(r'^\+\s*(?:self\.)?(\w+)\s*:\s*([\w][\w\[\], ]*)\s*=', re.MULTILINE)
_RE_JV_VAR  = re.compile(r'^\+\s*(int|float|double|String|bool|boolean|long|char)\s+(\w+)\s*[=;]', re.MULTILINE)
_RE_ISSUE   = re.compile(r'\b([A-Z]+-\d+|#\d+)\b')


def parse_files(diff):
    """list of (filepath, 'add'|'remove'|'modify')"""
    if not isinstance(diff, str): return []
    out = []
    for m in _RE_FILE.finditer(diff):
        a, b   = m.group(1), m.group(2)
        change = "add" if a == "/dev/null" else ("remove" if b == "/dev/null" else "modify")
        out.append((b if b != "/dev/null" else a, change))
    return out


def parse_author(diff):
    """(name, email) or (None, None)"""
    if not isinstance(diff, str): return None, None
    m = _RE_AUTHOR.search(diff)
    return (m.group(1), m.group(2)) if m else (None, None)


def parse_imports(diff):
    """set of top-level package names added in this diff"""
    if not isinstance(diff, str): return set()
    pkgs = set()
    for m in _RE_PY_IMP.finditer(diff):
        pkg = (m.group(1) or m.group(2) or "").split(".")[0]
        if pkg: pkgs.add(pkg)
    for m in _RE_JV_IMP.finditer(diff):
        pkgs.add(m.group(1).split(".")[0])
    return pkgs


def parse_classes(diff):
    if not isinstance(diff, str): return []
    return [m.group(1) for m in _RE_CLASS.finditer(diff)]


def parse_functions(diff):
    """list of (name, args_str, return_type_or_None)"""
    if not isinstance(diff, str): return []
    out = []
    for m in _RE_PY_FN.finditer(diff): out.append((m.group(1), m.group(2), m.group(3)))
    for m in _RE_JV_FN.finditer(diff): out.append((m.group(2), m.group(3), m.group(1)))
    return out


def parse_variables(diff):
    """list of (var_name, type_name)"""
    if not isinstance(diff, str): return []
    out = []
    for m in _RE_PY_VAR.finditer(diff): out.append((m.group(1), m.group(2).strip()))
    for m in _RE_JV_VAR.finditer(diff): out.append((m.group(2), m.group(1)))
    return out


def parse_issues(diff):
    """set of issue IDs from the commit message header"""
    if not isinstance(diff, str): return set()
    header = diff.split("diff --git")[0]   # message is before the first diff block
    return set(_RE_ISSUE.findall(header))


print("Parser functions ready.")

## 3 — `update_kg(G, row, diff_text, prev_cid)`

Called **after** prediction for each commit.
Adds all nodes and edges for that one commit to `G`.
Running counters on `AUTHOR` and `FILE` nodes make feature extraction O(1).

In [ ]:
def update_kg(G, row, diff, prev_cid=None):
    """
    Add one commit's knowledge to G.  Call AFTER prediction.

    G         – nx.MultiDiGraph, the running knowledge graph
    row       – pd.Series, one row from the time-sorted CSV
    diff      – str | None, raw git-show output for this commit
    prev_cid  – node-id string of the previous commit in this project, or None
    """
    cid = f"commit:{row.commit_id}"
    ts  = int(row.author_date)

    # ── TIER 1: Core ──────────────────────────────────────────────────

    G.add_node(cid, type="COMMIT", commit_id=row.commit_id,
               project=row.project, year=int(row.year), author_date=ts)

    tid = f"time:{ts}"
    if not G.has_node(tid):
        G.add_node(tid, type="TIME", datetime=ts)
    G.add_edge(cid, tid, rel="at_time")

    # INTERVAL between previous commit and this one
    if prev_cid is not None:
        prev_ts = G.nodes[prev_cid]["author_date"]
        iid = f"interval:{prev_cid}_{row.commit_id}"
        G.add_node(iid, type="INTERVAL", project=row.project)
        G.add_edge(iid, f"time:{prev_ts}", rel="begin")
        G.add_edge(iid, tid,              rel="end")
        G.add_edge(prev_cid, iid,          rel="duration")

    status = -1 if row.buggy else (1 if row.fix else 0)
    lid = f"label:{status}"
    if not G.has_node(lid):
        G.add_node(lid, type="LABEL", status=status)
    G.add_edge(cid, lid, rel="is")

    # ── TIER 2: Author ────────────────────────────────────────────────

    _, email = parse_author(diff)
    aid = f"author:{email or 'unknown'}"
    if not G.has_node(aid):
        G.add_node(aid, type="AUTHOR", email=email, commit_count=0, bug_count=0)
    G.add_edge(cid, aid, rel="by")
    G.nodes[aid]["commit_count"] += 1
    if row.buggy:
        G.nodes[aid]["bug_count"] += 1

    # Author interval: extend end pointer each time they commit
    auth_iid = f"interval:author:{email or 'unknown'}"
    if G.has_node(auth_iid):
        for _, t, d in list(G.out_edges(auth_iid, data=True)):
            if d.get("rel") == "end":
                G.remove_edge(auth_iid, t)
        G.add_edge(auth_iid, tid, rel="end")
    else:
        G.add_node(auth_iid, type="INTERVAL")
        G.add_edge(auth_iid, tid, rel="begin")
        G.add_edge(auth_iid, tid, rel="end")
        G.add_edge(aid, auth_iid, rel="duration")

    # ── TIER 2: Files, dirs, file types ──────────────────────────────

    files = parse_files(diff)
    for filepath, change_type in files:
        p    = Path(filepath)
        fid  = f"file:{filepath}"
        ftid = f"filetype:{p.suffix or 'none'}"

        if not G.has_node(fid):
            G.add_node(fid, type="FILE", name=p.name, path=filepath,
                       change_count=0, bug_count=0, authors=set())
        G.add_edge(cid, fid, rel=change_type)
        G.nodes[fid]["change_count"] += 1
        if row.buggy:
            G.nodes[fid]["bug_count"] += 1
        G.nodes[fid]["authors"].add(email or "unknown")

        if not G.has_node(ftid):
            G.add_node(ftid, type="FILE_TYPE", format=p.suffix)
        G.add_edge(fid, ftid, rel="type")

        parents = list(reversed(list(p.parents)))
        for i, part in enumerate(parents):
            if str(part) in (".", ""):
                continue
            did = f"dir:{part}"
            if not G.has_node(did):
                G.add_node(did, type="DIR", name=part.name, path=str(part))
            if i == len(parents) - 1:
                G.add_edge(fid, did, rel="parent")
            if i > 0:
                pdid = f"dir:{parents[i-1]}"
                if G.has_node(pdid):
                    G.add_edge(did, pdid, rel="parent")
                    G.add_edge(pdid, did, rel="child")

        # File interval: extend end pointer each time it is touched
        file_iid = f"interval:file:{filepath}"
        if G.has_node(file_iid):
            for _, t, d in list(G.out_edges(file_iid, data=True)):
                if d.get("rel") == "end":
                    G.remove_edge(file_iid, t)
            G.add_edge(file_iid, tid, rel="end")
        else:
            G.add_node(file_iid, type="INTERVAL")
            G.add_edge(file_iid, tid, rel="begin")
            G.add_edge(file_iid, tid, rel="end")
            G.add_edge(fid, file_iid, rel="duration")

    # ── TIER 3: External packages ─────────────────────────────────────

    for pkg in parse_imports(diff):
        pid = f"extpkg:{pkg}"
        if not G.has_node(pid):
            G.add_node(pid, type="EXTERNAL_PACKAGE", name=pkg)
        for fp, _ in files:
            G.add_edge(f"file:{fp}", pid, rel="imports")

    # ── TIER 4: Within-file entities ──────────────────────────────────

    fids = [f"file:{fp}" for fp, _ in files]

    for cls in parse_classes(diff):
        clid = f"class:{cls}"
        if not G.has_node(clid): G.add_node(clid, type="CLASS", name=cls)
        for fid in fids: G.add_edge(fid, clid, rel="contains")

    for fn, args, ret in parse_functions(diff):
        fnid   = f"func:{fn}"
        sig_id = f"sig:{fn}({args})->{ret}"
        if not G.has_node(fnid): G.add_node(fnid, type="FUNCTION", name=fn)
        G.add_node(sig_id, type="FUNCTION_SIGNATURE", args=args, returns=ret)
        G.add_edge(fnid, sig_id, rel="has_signature")
        for fid in fids: G.add_edge(fid, fnid, rel="contains")

    for var, dtype in parse_variables(diff):
        vid  = f"var:{var}"
        dtid = f"dtype:{dtype}"
        if not G.has_node(vid):  G.add_node(vid,  type="VARIABLE",  name=var)
        if not G.has_node(dtid): G.add_node(dtid, type="DATA_TYPE", id=dtype)
        G.add_edge(vid, dtid, rel="has_type")
        for fid in fids: G.add_edge(fid, vid, rel="contains")

    # ── TIER 5: Issues (from commit message header) ───────────────────

    for issue_id in parse_issues(diff):
        inode = f"issue:{issue_id}"
        if not G.has_node(inode): G.add_node(inode, type="ISSUE", id=issue_id)
        G.add_edge(cid, inode, rel="for")

    # ── TIER 5: Branch (placeholder — needs git repo) ─────────────────
    # import subprocess
    # branches = subprocess.check_output(
    #     ["git", "-C", str(REPO_PATH), "branch", "--contains", row.commit_id],
    #     text=True).strip().splitlines()
    # for b in branches:
    #     b = b.strip().lstrip("* ")
    #     if not b: continue
    #     bid = f"branch:{b}"
    #     if not G.has_node(bid): G.add_node(bid, type="BRANCH", id=b)
    #     G.add_edge(cid, bid, rel="in")


print("update_kg() ready.")

## 4 — `extract_kg_features(G, row, diff)`

Called **before** `update_kg` — `G` contains only past commits.
Returns a flat dict of floats ready to concatenate with the handcrafted feature vector.

In [ ]:
def extract_kg_features(G, row, diff):
    """
    Query G for features about this commit BEFORE it is added.
    All values default to 0.0 if the relevant history does not exist yet.
    """
    feats = {}

    # ── Project-level ─────────────────────────────────────────────────
    commit_nodes = [n for n, d in G.nodes(data=True) if d.get("type") == "COMMIT"]
    n = len(commit_nodes)
    feats["kg_project_commit_count"] = float(n)
    if n > 0:
        n_bug = sum(
            1 for cn in commit_nodes
            if any(d.get("rel") == "is" and v == "label:-1"
                   for _, v, d in G.out_edges(cn, data=True))
        )
        feats["kg_project_bug_rate"] = n_bug / n
    else:
        feats["kg_project_bug_rate"] = 0.0

    # ── Author-level ──────────────────────────────────────────────────
    _, email = parse_author(diff)
    aid = f"author:{email or 'unknown'}"
    ad  = G.nodes[aid] if G.has_node(aid) else {}
    ac  = ad.get("commit_count", 0)
    bc  = ad.get("bug_count",    0)
    feats["kg_author_commit_count"] = float(ac)
    feats["kg_author_bug_rate"]     = bc / ac if ac > 0 else 0.0

    # ── File-level ────────────────────────────────────────────────────
    change_counts, bug_rates, uniq_authors = [], [], []
    for fp, _ in parse_files(diff):
        fid = f"file:{fp}"
        if not G.has_node(fid):
            continue
        fd  = G.nodes[fid]
        cc  = fd.get("change_count", 0)
        bcc = fd.get("bug_count",    0)
        change_counts.append(cc)
        bug_rates.append(bcc / cc if cc > 0 else 0.0)
        uniq_authors.append(len(fd.get("authors", set())))

    feats["kg_file_change_count"]   = float(sum(change_counts))
    feats["kg_file_bug_rate"]       = sum(bug_rates)   / len(bug_rates)   if bug_rates   else 0.0
    feats["kg_file_unique_authors"] = sum(uniq_authors) / len(uniq_authors) if uniq_authors else 0.0

    return feats


print("extract_kg_features() ready.")

## 5 — Online Simulation Loop

Initialise an empty graph and a fresh online classifier.
The four-step loop runs over commits in strict chronological order.

In [ ]:
# Handcrafted features available from the CSV
HC_COLS = ["la", "ld", "nf", "nd", "ns", "ent", "ndev", "age", "nuc", "aexp", "arexp", "asexp"]

# KG features in fixed order
KG_COLS = [
    "kg_project_commit_count",
    "kg_project_bug_rate",
    "kg_author_commit_count",
    "kg_author_bug_rate",
    "kg_file_change_count",
    "kg_file_bug_rate",
    "kg_file_unique_authors",
]

# ── Initialise empty graph and model ─────────────────────────────────
G            = nx.MultiDiGraph()
model        = SGDClassifier(loss="log_loss", max_iter=1, warm_start=True, random_state=42)
scaler       = StandardScaler()
results      = []          # accumulates {commit_id, true_label, predicted}
prev_cid_map = {}          # project → node-id of last commit seen
WARMUP       = 50          # commits before we start predicting

print(f"Starting online simulation over {len(df):,} commits …")

In [ ]:
for i, row in df.iterrows():

    diff       = diff_map.get(row.commit_id)     # None if server CSV not loaded
    true_label = int(row.buggy)                  # 1 = buggy, 0 = clean

    # ── STEP 1: Extract KG features ── G knows nothing about this commit yet ──
    kg   = extract_kg_features(G, row, diff)
    hc   = [float(row[c]) for c in HC_COLS]
    x    = np.array(hc + [kg[k] for k in KG_COLS], dtype=float)

    # ── STEP 2: Predict ───────────────────────────────────────────────────────
    predicted = None
    if i >= WARMUP:
        x_sc      = scaler.transform(x.reshape(1, -1))
        predicted = int(model.predict(x_sc)[0])

    # ── STEP 3: Record result ─────────────────────────────────────────────────
    results.append({
        "commit_id":   row.commit_id,
        "author_date": row.author_date,
        "true_label":  true_label,
        "predicted":   predicted,
    })

    # ── STEP 4: Update KG ─────────────────────────────────────────────────────
    prev_cid = prev_cid_map.get(row.project)
    update_kg(G, row, diff, prev_cid=prev_cid)
    prev_cid_map[row.project] = f"commit:{row.commit_id}"

    # ── Online model update ───────────────────────────────────────────────────
    scaler.partial_fit(x.reshape(1, -1))
    x_sc = scaler.transform(x.reshape(1, -1))
    model.partial_fit(x_sc, [true_label], classes=[0, 1])

    if (i + 1) % 1000 == 0:
        done = i + 1
        print(f"  {done:,} / {len(df):,} commits processed …")

print(f"\nDone. G has {G.number_of_nodes():,} nodes and {G.number_of_edges():,} edges.")

## 6 — Evaluation

In [ ]:
results_df = pd.DataFrame(results).dropna(subset=["predicted"])
y_true = results_df.true_label.astype(int)
y_pred = results_df.predicted.astype(int)

print(f"Evaluated on {len(results_df):,} commits (after {WARMUP}-commit warm-up)\n")
print(classification_report(y_true, y_pred, target_names=["clean", "buggy"]))
print("Confusion matrix (rows=true, cols=pred):")
print(confusion_matrix(y_true, y_pred))

## 7 — Graph Inspection & Export

In [ ]:
# Node / edge type breakdown
node_counts = pd.Series(nx.get_node_attributes(G, "type")).value_counts()
edge_counts = pd.Series([d.get("rel") for _, _, d in G.edges(data=True)]).value_counts()

print(f"Nodes : {G.number_of_nodes():,}   Edges : {G.number_of_edges():,}\n")
print("── Node types ──────────────")
print(node_counts.to_string())
print("\n── Edge relation types ─────")
print(edge_counts.head(20).to_string())

In [ ]:
# Sample query: all outgoing edges from the last commit
last_cid = f"commit:{df.commit_id.iloc[-1]}"
print(f"Neighbourhood of {last_cid}:\n")
for _, nb, d in G.out_edges(last_cid, data=True):
    print(f"  --[{d['rel']}]--> {nb}  ({G.nodes[nb].get('type', '?')})")

In [ ]:
OUTPUT_DIR = Path("../../outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# GraphML is not compatible with Python sets — convert authors to string first
for _, d in G.nodes(data=True):
    if "authors" in d:
        d["authors"] = ",".join(sorted(d["authors"]))

graphml_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.graphml"
nx.write_graphml(G, graphml_path)
print(f"GraphML : {graphml_path}")

json_path = OUTPUT_DIR / f"{PROJECT_NAME}_kg.json"
with open(json_path, "w") as f:
    json.dump(nx.node_link_data(G), f, indent=2)
print(f"JSON    : {json_path}")

results_path = OUTPUT_DIR / f"{PROJECT_NAME}_online_results.csv"
results_df.to_csv(results_path, index=False)
print(f"Results : {results_path}")